# LeetCode 406 — Queue Reconstruction by Height bằng Python

Link đề: https://leetcode.com/problems/queue-reconstruction-by-height/ · Độ khó: **Medium** · Chủ đề: Array, Sorting, Greedy, Binary Indexed Tree, Segment Tree

Skeleton LeetCode:

```python
class Solution:
    def reconstructQueue(self, people: List[List[int]]) -> List[List[int]]:
        ...
```

Ghi chú: phần phát biểu đề được dịch sang tiếng Việt từ đề gốc. Bài này kiểm tra đáp án bằng **hàm kiểm tra tính hợp lệ** của hàng đợi (đếm số người đứng trước có chiều cao `>=`) và bằng vét cạn mọi hoán vị trên input nhỏ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi người là cặp `[h, k]`: chiều cao `h` và có **đúng `k`** người đứng **trước** mình có chiều cao **lớn hơn hoặc bằng** `h`. Cho danh sách bị xáo trộn, hãy dựng lại hàng đợi.
2. Nắm công cụ Python: `list.sort(key=...)` với khóa nhiều tầng và dấu trừ, `list.insert(index, x)`, `lambda`, danh sách `None` làm "ô trống", `sorted`, `itertools.permutations`, cây Fenwick (BIT) ở mức cơ bản.
3. Đi từ cách **thử mọi hoán vị** → **xếp người cao trước rồi chèn vào vị trí `k`** → **xếp người thấp trước, đặt vào ô trống thứ `k`** → **dùng BIT để tìm ô trống thứ `k` nhanh** → **dựng từ đầu hàng: chọn người có `k = 0` thấp nhất**.
4. Hiểu mấu chốt: người **thấp hơn không ảnh hưởng** đến số đếm của người cao hơn. Vì vậy ta xử lý theo chiều cao và mỗi người chỉ cần "nhìn" những người cao hơn/bằng đã đặt.
5. Tránh các lỗi kinh điển: sắp xếp sai thứ tự phụ (`k` giảm thay vì tăng, hoặc ngược lại), chèn theo `h` thay vì `k`, đếm người thấp hơn, làm biến đổi input, quên rằng người cao bằng nhau vẫn được tính.

## 2. Đề bài & ví dụ

Cho mảng `people`, trong đó `people[i] = [h_i, k_i]` mô tả người thứ `i` trong một hàng đợi (thứ tự đang bị xáo trộn): người đó cao `h_i` và có **đúng `k_i`** người **đứng trước** mình có chiều cao **>= `h_i`**.

Hãy dựng lại và trả về hàng đợi `queue`, với `queue[0]` là người đứng đầu hàng.

**Ví dụ 1:** `people = [[7,0],[4,4],[7,1],[5,0],[6,1],[5,2]]` → `[[5,0],[7,0],[5,2],[6,1],[4,4],[7,1]]`.

**Ví dụ 2:** `people = [[6,0],[5,0],[4,0],[3,2],[2,2],[1,4]]` → `[[4,0],[5,0],[2,2],[3,2],[1,4],[6,0]]`.

**Ràng buộc:**

- `1 <= people.length <= 2000`
- `0 <= h_i <= 10^6`
- `0 <= k_i < people.length`
- Đảm bảo hàng đợi **có thể** dựng lại.

In [1]:
from itertools import permutations
import random
import time
from typing import List

examples = [
    ([[7, 0], [4, 4], [7, 1], [5, 0], [6, 1], [5, 2]], [[5, 0], [7, 0], [5, 2], [6, 1], [4, 4], [7, 1]]),
    ([[6, 0], [5, 0], [4, 0], [3, 2], [2, 2], [1, 4]], [[4, 0], [5, 0], [2, 2], [3, 2], [1, 4], [6, 0]]),
]


def is_valid_queue(queue):
    """Kiểm tra hàng đợi: với mỗi vị trí i, số người đứng trước có chiều cao >= h đúng bằng k."""
    for i, (h, k) in enumerate(queue):
        cnt = sum(1 for j in range(i) if queue[j][0] >= h)
        if cnt != k:
            return False
    return True


for people, expected in examples:
    print("đáp án đề:", expected, "| hợp lệ?", is_valid_queue(expected))
    assert is_valid_queue(expected) and sorted(people) == sorted(expected)

đáp án đề: [[5, 0], [7, 0], [5, 2], [6, 1], [4, 4], [7, 1]] | hợp lệ? True
đáp án đề: [[4, 0], [5, 0], [2, 2], [3, 2], [1, 4], [6, 0]] | hợp lệ? True


### 2.1. Đọc kỹ đề

- `k` đếm người đứng **trước** có chiều cao **lớn hơn hoặc bằng** `h` — **gồm cả người cao bằng**. Người thấp hơn **không** được đếm.
- Đề không nói "chỉ có một đáp án", nhưng với dữ liệu hợp lệ, hàng đợi dựng lại là **duy nhất** (ta sẽ kiểm chứng bằng vét cạn ở mục 12). Dù vậy, để an toàn, ta kiểm tra đáp án bằng **hàm hợp lệ** `is_valid_queue` ở trên chứ không chỉ so sánh cứng.
- Input có thể bị thay đổi nếu ta sắp xếp tại chỗ (`people.sort`). LeetCode không kiểm tra điều này, nhưng khi tự test nên **sao chép** input.

## 3. Kiến thức nền

### 3.1. `sort` với `key` nhiều tầng và dấu trừ

`list.sort(key=f)` sắp xếp theo giá trị `f(x)` tăng dần. Nếu `f` trả về **tuple** thì so sánh từng phần tử theo thứ tự. Để sắp **giảm dần** theo một thành phần số, đặt dấu trừ trước nó: `key=lambda p: (-p[0], p[1])` nghĩa là `h` giảm dần, nếu `h` bằng nhau thì `k` tăng dần.

In [2]:
people = [[7, 0], [4, 4], [7, 1], [5, 0], [6, 1], [5, 2]]
print(sorted(people, key=lambda p: (-p[0], p[1])))     # h giảm, k tăng
print(sorted(people, key=lambda p: (p[0], -p[1])))     # h tăng, k giảm

[[7, 0], [7, 1], [6, 1], [5, 0], [5, 2], [4, 4]]
[[4, 4], [5, 2], [5, 0], [6, 1], [7, 1], [7, 0]]


### 3.2. `list.insert(i, x)`

`a.insert(i, x)` chèn `x` vào **vị trí `i`**, các phần tử từ `i` trở đi dịch sang phải. Nếu `i >= len(a)` thì chèn ở cuối. Chi phí `O(len(a))` vì phải dịch phần tử.

In [3]:
a = ["A", "B", "C"]
a.insert(1, "X"); print(a)
a.insert(0, "Y"); print(a)
a.insert(99, "Z"); print(a)

['A', 'X', 'B', 'C']
['Y', 'A', 'X', 'B', 'C']
['Y', 'A', 'X', 'B', 'C', 'Z']


### 3.3. "Ô trống" bằng `None` và đếm ô trống

Ta có thể tạo hàng `[None] * n` gồm `n` ô trống rồi lần lượt đặt người vào. Để tìm "ô trống thứ `k`", duyệt và đếm các ô còn `None`.

In [4]:
slots = [None, "B", None, None, "E"]
cnt = 0
for i, v in enumerate(slots):
    if v is None:
        if cnt == 1:                                    # ô trống thứ 1 (đếm từ 0)
            print("ô trống thứ 1 nằm ở chỉ số", i)
            break
        cnt += 1

ô trống thứ 1 nằm ở chỉ số 2


### 3.4. `itertools.permutations`

`permutations(items)` sinh mọi cách xếp thứ tự (`n!` cách). Với `n = 6` có `720`, với `n = 8` có `40320`. Ta dùng nó làm **oracle** cho input nhỏ.

In [5]:
print(list(permutations([1, 2, 3])))
from math import factorial
print([factorial(n) for n in range(1, 11)])

[(1, 2, 3), (1, 3, 2), (2, 1, 3), (2, 3, 1), (3, 1, 2), (3, 2, 1)]
[1, 2, 6, 24, 120, 720, 5040, 40320, 362880, 3628800]


### 3.5. Cây Fenwick (Binary Indexed Tree) — cực ngắn gọn

BIT lưu mảng đếm để làm hai việc trong `O(log n)`: **cộng 1 đơn vị tại vị trí `i`** và **tính tổng tiền tố**. Ở Cách 4 ta dùng nó để quản lý số ô trống: ban đầu mọi ô có giá trị `1` (trống), khi đặt người vào ô thì trừ đi `1`. Để tìm "ô trống thứ `k`" ta đi xuống cây (binary lifting). Chi tiết có ở mục 7.

In [6]:
class Fenwick:
    def __init__(self, n):
        self.n = n
        self.t = [0] * (n + 1)

    def add(self, i, v):                 # i đếm từ 1
        while i <= self.n:
            self.t[i] += v
            i += i & -i

    def prefix(self, i):                 # tổng a[1..i]
        s = 0
        while i > 0:
            s += self.t[i]
            i -= i & -i
        return s


f = Fenwick(5)
for i in range(1, 6):
    f.add(i, 1)
f.add(2, -1)                              # ô 2 bị chiếm
print([f.prefix(i) for i in range(6)])   # [0, 1, 1, 2, 3, 4]

[0, 1, 1, 2, 3, 4]


## 4. Ý tưởng nền tảng: tính tay

Hãy nhìn ví dụ 1: `[[7,0],[4,4],[7,1],[5,0],[6,1],[5,2]]`.

**Quan sát then chốt:** số `k` của một người **chỉ phụ thuộc vào những người cao hơn hoặc bằng**. Người thấp hơn có đứng trước hay sau cũng không đổi `k` của họ.

Xếp từ người **cao nhất** xuống:

1. Cao `7`: `[7,0]` và `[7,1]`. Người `[7,0]` đứng đầu. `[7,1]` có đúng 1 người `>=7` đứng trước ⇒ sau `[7,0]`. Hàng: `[7,0], [7,1]`.
2. Cao `6`: `[6,1]` cần **đúng 1** người `>= 6` trước mình ⇒ chèn vào **vị trí 1**. Hàng: `[7,0], [6,1], [7,1]`.
3. Cao `5`: `[5,0]` chèn vị trí `0`; `[5,2]` chèn vị trí `2`. Làm theo thứ tự `k` tăng: `[5,0], [7,0], [5,2], [6,1], [7,1]`... để ý `[5,2]` chèn ở vị trí `2` sau khi `[5,0]` đã vào.
4. Cao `4`: `[4,4]` chèn vị trí `4`.

Kết quả cuối `[[5,0],[7,0],[5,2],[6,1],[4,4],[7,1]]` đúng với đáp án đề. Ở mỗi bước, tất cả người đã có mặt đều **cao hơn hoặc bằng** người đang chèn, nên chèn tại **vị trí `k`** nghĩa là đúng `k` người trước mặt.

## 5. Cách 1 — Brute Force: thử mọi hoán vị

**Ý tưởng:** sinh mọi cách sắp xếp `people`, trả về cách đầu tiên thỏa `is_valid_queue`. Vì đề đảm bảo tồn tại, luôn tìm được.

In [7]:
def reconstruct_brute(people):
    for perm in permutations(people):
        if is_valid_queue(perm):
            return [list(p) for p in perm]
    return None


for people, expected in examples[:1]:
    got = reconstruct_brute(people)
    print(got)
    assert got == expected and is_valid_queue(got)
print("ví dụ 2 (6 người, 720 hoán vị):", reconstruct_brute(examples[1][0]) == examples[1][1])

[[5, 0], [7, 0], [5, 2], [6, 1], [4, 4], [7, 1]]
ví dụ 2 (6 người, 720 hoán vị): True


**Độ phức tạp:** `n!` hoán vị, mỗi lần kiểm tra `O(n^2)` ⇒ Time `O(n! * n^2)`, Space `O(n)`.

**Nút thắt:** `n = 2000` thì `n!` là con số khổng lồ; ngay `n = 12` đã hơn `479` triệu hoán vị.

In [8]:
for n in [5, 8, 10, 12, 15]:
    print(f"n={n:>2}: {factorial(n):,} hoán vị")

n= 5: 120 hoán vị
n= 8: 40,320 hoán vị
n=10: 3,628,800 hoán vị
n=12: 479,001,600 hoán vị
n=15: 1,307,674,368,000 hoán vị


## 6. Ý tưởng thuật toán / bước đột phá

Hai tính chất:

- **Người thấp không ảnh hưởng người cao:** thêm một người thấp hơn `h` vào bất cứ chỗ nào không đổi số đếm `>= h` của người cao `h`.
- Vì vậy, nếu ta đã xếp **đúng** hàng cho tất cả người cao hơn `h`, thì khi **chèn** một người cao `h` vào vị trí `k`, anh ta có đúng `k` người cao-hơn-hoặc-bằng đứng trước, và những người đã xếp không bị ảnh hưởng.

Chiến lược: **sắp xếp theo `h` giảm dần**; với `h` bằng nhau thì **`k` tăng dần** (để người có `k` nhỏ hơn vào trước, người có `k` lớn hơn chèn sau và "đẩy" qua đúng chỗ). Rồi lần lượt `ans.insert(k, person)`.

Vì sao `k` tăng dần với `h` bằng nhau? Hai người cùng cao `h` có `k1 < k2`: người `k1` phải đứng **trước** người `k2` (nếu không, người `k1` sẽ có người cao bằng đứng trước nhiều hơn). Chèn `k1` trước ⇒ khi chèn `k2` vào vị trí `k2 > k1` thì nó nằm sau `k1`. Ngược lại nếu chèn `k2` trước, vị trí `k2` có thể vượt độ dài hiện tại.

## 7. Các cách tiếp theo

### Cách 2 — Sắp xếp người cao trước, chèn vào vị trí `k`

**Giải thích từng đoạn:**

1. `sorted(people, key=lambda p: (-p[0], p[1]))`: chiều cao giảm dần, cùng chiều cao thì `k` tăng dần.
2. `ans = []`.
3. Với mỗi `[h, k]`: `ans.insert(k, [h, k])`.
4. Trả về `ans`.

In [9]:
def reconstruct_insert(people):
    ans = []
    for h, k in sorted(people, key=lambda p: (-p[0], p[1])):
        ans.insert(k, [h, k])
    return ans


for people, expected in examples:
    got = reconstruct_insert(people)
    print(got)
    assert got == expected

[[5, 0], [7, 0], [5, 2], [6, 1], [4, 4], [7, 1]]
[[4, 0], [5, 0], [2, 2], [3, 2], [1, 4], [6, 0]]


In [10]:
# Code nộp LeetCode — Cách 2 (O(n^2) time vì insert dịch phần tử, O(n) space)
class SolutionInsert:
    def reconstructQueue(self, people: List[List[int]]) -> List[List[int]]:
        people = sorted(people, key=lambda x: (-x[0], x[1]))
        ans = []
        for p in people:
            ans.insert(p[1], p)
        return ans


print([SolutionInsert().reconstructQueue([p[:] for p in people]) == exp for people, exp in examples])

[True, True]


**Độ phức tạp Cách 2:** sắp xếp `O(n log n)`; `n` lần `insert`, mỗi lần dịch tối đa `n` phần tử ⇒ `O(n^2)`. Với `n = 2000` chỉ là `4 * 10^6` thao tác (bước dịch của `list.insert` là C thuần, rất nhanh). Space `O(n)`.

### Cách 3 — Sắp xếp người thấp trước, đặt vào "ô trống thứ `k`"

**Ý tưởng đảo chiều:** chuẩn bị `n` ô trống. Xử lý người **thấp nhất trước**. Khi xử lý người `[h, k]`, những người sẽ đứng trước họ mà có chiều cao `>= h` là những người **chưa được xử lý** (cao hơn, hoặc bằng nhưng xử lý sau) — họ sẽ lấp các **ô trống** phía trước. Vậy `[h, k]` phải đặt vào **ô trống thứ `k`** (đếm từ `0`) trong các ô trống hiện tại.

Với chiều cao bằng nhau: phải xử lý `k` **giảm dần**. Nếu người `k` nhỏ xử lý trước thì nó chiếm một ô trống phía trước và người cùng cao có `k` lớn sẽ thấy ô đó không còn là "trống" (không được đếm) — dẫn đến đếm thiếu. Nên người `k` lớn vào trước, người `k` nhỏ vào sau (khi đó ô trống của nó nằm trước người `k` lớn).

**Giải thích từng đoạn:**

1. `sorted(people, key=lambda p: (p[0], -p[1]))`: `h` tăng, cùng `h` thì `k` giảm.
2. `ans = [None] * n`.
3. Với mỗi `[h, k]`: duyệt `ans`, đếm ô `None`; ô trống thứ `k` thì gán và dừng.

In [11]:
def reconstruct_slots(people):
    n = len(people)
    ans = [None] * n
    for h, k in sorted(people, key=lambda p: (p[0], -p[1])):
        cnt = 0
        for i in range(n):
            if ans[i] is None:
                if cnt == k:
                    ans[i] = [h, k]
                    break
                cnt += 1
    return ans


for people, expected in examples:
    got = reconstruct_slots(people)
    print(got)
    assert got == expected

[[5, 0], [7, 0], [5, 2], [6, 1], [4, 4], [7, 1]]
[[4, 0], [5, 0], [2, 2], [3, 2], [1, 4], [6, 0]]


In [12]:
# Code nộp LeetCode — Cách 3 (O(n^2) time, O(n) space)
class SolutionSlots:
    def reconstructQueue(self, people: List[List[int]]) -> List[List[int]]:
        n = len(people)
        ans = [None] * n
        for h, k in sorted(people, key=lambda x: (x[0], -x[1])):
            cnt = 0
            for i in range(n):
                if ans[i] is None:
                    if cnt == k:
                        ans[i] = [h, k]
                        break
                    cnt += 1
        return ans


print([SolutionSlots().reconstructQueue([p[:] for p in people]) == exp for people, exp in examples])

[True, True]


**Độ phức tạp Cách 3:** mỗi người quét tối đa `n` ô ⇒ Time `O(n^2)`, Space `O(n)`.

### Cách 4 — Cùng ý tưởng ô trống nhưng dùng BIT (Fenwick)

**Điểm nghẽn của Cách 3** là quét tuyến tính để tìm ô trống thứ `k`. Dùng BIT lưu "ô `i` còn trống (1) hay đã chiếm (0)":

- `find_kth(k)` tìm vị trí nhỏ nhất `pos` sao cho `prefix(pos) = k + 1` bằng **binary lifting**: từ bit lớn nhất, thử nhảy `pos + 2^j` nếu tổng tại đó `< k + 1` thì cộng vào.
- Sau khi đặt, `add(pos, -1)`.

Khởi tạo BIT từ mảng toàn `1` trong `O(n)`: `t[i] = i & -i` (số phần tử mà nút `i` quản lý).

In [13]:
def reconstruct_bit(people):
    n = len(people)
    t = [0] * (n + 1)
    for i in range(1, n + 1):
        t[i] = i & -i                         # BIT của mảng toàn 1
    top = 1
    while top * 2 <= n:
        top *= 2

    def find_kth(rank):                       # vị trí (1-based) của ô trống thứ `rank` (rank đếm từ 1)
        pos, step = 0, top
        while step:
            nxt = pos + step
            if nxt <= n and t[nxt] < rank:
                pos = nxt
                rank -= t[nxt]
            step >>= 1
        return pos + 1

    def remove(i):
        while i <= n:
            t[i] -= 1
            i += i & -i

    ans = [None] * n
    for h, k in sorted(people, key=lambda p: (p[0], -p[1])):
        pos = find_kth(k + 1)
        ans[pos - 1] = [h, k]
        remove(pos)
    return ans


for people, expected in examples:
    got = reconstruct_bit(people)
    print(got)
    assert got == expected

[[5, 0], [7, 0], [5, 2], [6, 1], [4, 4], [7, 1]]
[[4, 0], [5, 0], [2, 2], [3, 2], [1, 4], [6, 0]]


In [14]:
# Code nộp LeetCode — Cách 4 (O(n log n) time, O(n) space)
class SolutionBIT:
    def reconstructQueue(self, people: List[List[int]]) -> List[List[int]]:
        n = len(people)
        t = [0] * (n + 1)
        for i in range(1, n + 1):
            t[i] = i & -i
        top = 1
        while top * 2 <= n:
            top *= 2
        ans = [None] * n
        for h, k in sorted(people, key=lambda x: (x[0], -x[1])):
            rank, pos, step = k + 1, 0, top
            while step:
                nxt = pos + step
                if nxt <= n and t[nxt] < rank:
                    pos = nxt
                    rank -= t[nxt]
                step >>= 1
            pos += 1
            ans[pos - 1] = [h, k]
            i = pos
            while i <= n:
                t[i] -= 1
                i += i & -i
        return ans


print([SolutionBIT().reconstructQueue([p[:] for p in people]) == exp for people, exp in examples])

[True, True]


**Độ phức tạp Cách 4:** mỗi người tốn `O(log n)` (tìm + cập nhật) ⇒ Time `O(n log n)`, Space `O(n)`. Với `n <= 2000`, Cách 2 vẫn nhanh hơn trên thực tế vì `list.insert` rất nhẹ; BIT mới đáng giá khi `n` lớn (ví dụ `10^5`).

### Cách 5 — Dựng từ đầu hàng: chọn người `k = 0` thấp nhất

**Ý tưởng khác:** nhìn **người đứng đầu hàng**. Người đầu hàng phải có `k = 0`. Trong những người có `k = 0` (còn lại), ai đứng đầu? Người **thấp nhất** (nếu chọn người cao hơn đứng trước, người thấp hơn có `k = 0` đứng sau sẽ bị tính thêm 1). Sau khi chọn người `[h, 0]` đặt lên đầu, mọi người còn lại có chiều cao `<= h` sẽ có **thêm một người `>=` đứng trước** nên `k` của họ phải **giảm đi 1** (phần còn lại của bài toán). Lặp lại.

**Giải thích từng đoạn:**

1. `rest` là danh sách người chưa đặt (bản sao có thể sửa `k`).
2. Mỗi vòng: trong `rest`, chọn người có `k == 0` và `h` nhỏ nhất.
3. Thêm vào kết quả; xóa khỏi `rest`; với mọi người còn lại có `h <= h_đã chọn`: `k -= 1`.
4. Ghi lại cặp `[h, k]` **gốc** của người đã chọn.

In [15]:
def reconstruct_front(people):
    rest = [[h, k, k] for h, k in people]          # [h, k còn lại, k gốc]
    ans = []
    while rest:
        best = None
        for idx, (h, kk, k0) in enumerate(rest):
            if kk == 0 and (best is None or h < rest[best][0]):
                best = idx
        h, _, k0 = rest.pop(best)
        ans.append([h, k0])
        for item in rest:
            if item[0] <= h:
                item[1] -= 1
    return ans


for people, expected in examples:
    got = reconstruct_front(people)
    print(got)
    assert got == expected

[[5, 0], [7, 0], [5, 2], [6, 1], [4, 4], [7, 1]]
[[4, 0], [5, 0], [2, 2], [3, 2], [1, 4], [6, 0]]


In [16]:
# Code nộp LeetCode — Cách 5 (O(n^2) time, O(n) space)
class SolutionFront:
    def reconstructQueue(self, people: List[List[int]]) -> List[List[int]]:
        rest = [[h, k, k] for h, k in people]
        ans = []
        while rest:
            best = -1
            for idx, item in enumerate(rest):
                if item[1] == 0 and (best == -1 or item[0] < rest[best][0]):
                    best = idx
            h, _, k0 = rest.pop(best)
            ans.append([h, k0])
            for item in rest:
                if item[0] <= h:
                    item[1] -= 1
        return ans


print([SolutionFront().reconstructQueue([p[:] for p in people]) == exp for people, exp in examples])

[True, True]


**Độ phức tạp Cách 5:** mỗi vòng quét `O(n)` hai lần ⇒ Time `O(n^2)`, Space `O(n)`. Cách này giúp thấy bài toán dưới một lăng kính khác: "mỗi lần xác định được **một** người ở **đầu** hàng".

## 8. TRACE từng bước

Trace Cách 2 với `people = [[7,0],[4,4],[7,1],[5,0],[6,1],[5,2]]`. Sau khi sắp xếp (`h` giảm, `k` tăng): `[7,0],[7,1],[6,1],[5,0],[5,2],[4,4]`.

| bước | người | chèn tại | hàng sau khi chèn |
|---|---|---|---|
| 1 | `[7,0]` | 0 | `[7,0]` |
| 2 | `[7,1]` | 1 | `[7,0],[7,1]` |
| 3 | `[6,1]` | 1 | `[7,0],[6,1],[7,1]` |
| 4 | `[5,0]` | 0 | `[5,0],[7,0],[6,1],[7,1]` |
| 5 | `[5,2]` | 2 | `[5,0],[7,0],[5,2],[6,1],[7,1]` |
| 6 | `[4,4]` | 4 | `[5,0],[7,0],[5,2],[6,1],[4,4],[7,1]` |

Hàm `trace_insert` in lại bảng này; `trace_slots` in các ô trống ở Cách 3; `trace_front` in từng lần chọn người đầu hàng ở Cách 5.

In [17]:
def trace_insert(people):
    order = sorted(people, key=lambda p: (-p[0], p[1]))
    print("sau khi sắp xếp:", order)
    ans = []
    for step, (h, k) in enumerate(order, 1):
        ans.insert(k, [h, k])
        print(f"bước {step}: chèn [{h},{k}] tại {k} -> {ans}")
    print("hợp lệ?", is_valid_queue(ans))
    return ans


trace_insert([[7, 0], [4, 4], [7, 1], [5, 0], [6, 1], [5, 2]])

sau khi sắp xếp: [[7, 0], [7, 1], [6, 1], [5, 0], [5, 2], [4, 4]]
bước 1: chèn [7,0] tại 0 -> [[7, 0]]
bước 2: chèn [7,1] tại 1 -> [[7, 0], [7, 1]]
bước 3: chèn [6,1] tại 1 -> [[7, 0], [6, 1], [7, 1]]
bước 4: chèn [5,0] tại 0 -> [[5, 0], [7, 0], [6, 1], [7, 1]]
bước 5: chèn [5,2] tại 2 -> [[5, 0], [7, 0], [5, 2], [6, 1], [7, 1]]
bước 6: chèn [4,4] tại 4 -> [[5, 0], [7, 0], [5, 2], [6, 1], [4, 4], [7, 1]]
hợp lệ? True


[[5, 0], [7, 0], [5, 2], [6, 1], [4, 4], [7, 1]]

In [18]:
def trace_slots(people):
    n = len(people)
    ans = [None] * n
    for step, (h, k) in enumerate(sorted(people, key=lambda p: (p[0], -p[1])), 1):
        empties = [i for i in range(n) if ans[i] is None]
        pos = empties[k]
        ans[pos] = [h, k]
        print(f"bước {step}: [{h},{k}] ô trống {empties} -> ô thứ {k} là {pos}")
        print("        hàng:", ans)
    return ans


trace_slots([[7, 0], [4, 4], [7, 1], [5, 0], [6, 1], [5, 2]])

bước 1: [4,4] ô trống [0, 1, 2, 3, 4, 5] -> ô thứ 4 là 4
        hàng: [None, None, None, None, [4, 4], None]
bước 2: [5,2] ô trống [0, 1, 2, 3, 5] -> ô thứ 2 là 2
        hàng: [None, None, [5, 2], None, [4, 4], None]
bước 3: [5,0] ô trống [0, 1, 3, 5] -> ô thứ 0 là 0
        hàng: [[5, 0], None, [5, 2], None, [4, 4], None]
bước 4: [6,1] ô trống [1, 3, 5] -> ô thứ 1 là 3
        hàng: [[5, 0], None, [5, 2], [6, 1], [4, 4], None]
bước 5: [7,1] ô trống [1, 5] -> ô thứ 1 là 5
        hàng: [[5, 0], None, [5, 2], [6, 1], [4, 4], [7, 1]]
bước 6: [7,0] ô trống [1] -> ô thứ 0 là 1
        hàng: [[5, 0], [7, 0], [5, 2], [6, 1], [4, 4], [7, 1]]


[[5, 0], [7, 0], [5, 2], [6, 1], [4, 4], [7, 1]]

In [19]:
def trace_front(people):
    rest = [[h, k, k] for h, k in people]
    ans = []
    step = 0
    while rest:
        step += 1
        cands = [(h, k0) for h, kk, k0 in rest if kk == 0]
        best = min(range(len(rest)), key=lambda i: (rest[i][0] if rest[i][1] == 0 else 10**9))
        h, _, k0 = rest.pop(best)
        ans.append([h, k0])
        dec = [(x[0], x[2]) for x in rest if x[0] <= h]
        for item in rest:
            if item[0] <= h:
                item[1] -= 1
        print(f"lượt {step}: ứng viên k=0 {cands} -> chọn thấp nhất [{h},{k0}] | giảm k của {dec}")
    print("kết quả:", ans)
    return ans


trace_front([[7, 0], [4, 4], [7, 1], [5, 0], [6, 1], [5, 2]])

lượt 1: ứng viên k=0 [(7, 0), (5, 0)] -> chọn thấp nhất [5,0] | giảm k của [(4, 4), (5, 2)]
lượt 2: ứng viên k=0 [(7, 0)] -> chọn thấp nhất [7,0] | giảm k của [(4, 4), (7, 1), (6, 1), (5, 2)]
lượt 3: ứng viên k=0 [(7, 1), (6, 1), (5, 2)] -> chọn thấp nhất [5,2] | giảm k của [(4, 4)]
lượt 4: ứng viên k=0 [(7, 1), (6, 1)] -> chọn thấp nhất [6,1] | giảm k của [(4, 4)]
lượt 5: ứng viên k=0 [(4, 4), (7, 1)] -> chọn thấp nhất [4,4] | giảm k của []
lượt 6: ứng viên k=0 [(7, 1)] -> chọn thấp nhất [7,1] | giảm k của []
kết quả: [[5, 0], [7, 0], [5, 2], [6, 1], [4, 4], [7, 1]]


[[5, 0], [7, 0], [5, 2], [6, 1], [4, 4], [7, 1]]

## 9. Code nộp LeetCode

Lời giải chính là **Cách 2** (sắp xếp người cao trước rồi chèn): ngắn nhất và dễ chứng minh.

In [20]:
class Solution:
    def reconstructQueue(self, people: List[List[int]]) -> List[List[int]]:
        people = sorted(people, key=lambda x: (-x[0], x[1]))   # cao -> thấp; cùng cao: k tăng
        ans = []
        for p in people:
            ans.insert(p[1], p)                                 # trước mặt đã toàn người cao hơn/bằng
        return ans


for people, expected in examples:
    got = Solution().reconstructQueue([p[:] for p in people])
    print(got)
    assert got == expected and is_valid_queue(got)

[[5, 0], [7, 0], [5, 2], [6, 1], [4, 4], [7, 1]]
[[4, 0], [5, 0], [2, 2], [3, 2], [1, 4], [6, 0]]


## 10. Vì sao đúng

1. **Bất biến:** sau khi xử lý tất cả người có chiều cao `>= H`, hàng `ans` là hàng hợp lệ **riêng cho nhóm đó** (mỗi người có đúng `k` người cao-hơn-hoặc-bằng đứng trước **trong nhóm**, mà nhóm là tất cả người `>= H`).
2. **Bước chèn:** khi chèn người `[h, k]` (chiều cao nhỏ nhất hiện tại, hoặc bằng chiều cao đã có), mọi người đang có trong `ans` đều cao `>= h`. Chèn ở vị trí `k` ⇒ phía trước đúng `k` người, mọi người đó đều được đếm ⇒ `k` đúng.
3. **Không phá người cũ:** người mới có `h` **không lớn hơn** bất kỳ ai trong `ans`, nên không được đếm vào `k` của họ. Với người **cùng** `h` đã có trước đó: họ có `k' <= k` nhờ sắp xếp `k` tăng dần. Trong hàng hiện tại (chỉ toàn người `>= h`), người `k'` có đúng `k'` người đứng trước, tức đang ở chỉ số `k'` ngay lúc được chèn và chỉ bị đẩy về sau bởi các lần chèn tiếp theo. Người mới chèn tại `k >= k'`: nếu `k > k'` thì nằm sau họ; nếu `k == k'` thì chen vào đúng chỗ họ và đẩy họ về sau — trong cả hai trường hợp không có ai chen lên **trước** họ, nên số người đứng trước họ không tăng.
4. **Tính duy nhất:** hàng đợi dựng lại là duy nhất: người đứng cuối cùng trong số người cao nhất bị ràng buộc hoàn toàn, quy nạp theo nhóm chiều cao giảm dần. Mục 12 kiểm chứng bằng vét cạn mọi hoán vị.

## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi hoán vị | `O(n! * n^2)` | `O(n)` |
| 2. Cao trước + `insert` | **`O(n^2)`** (sắp xếp `O(n log n)`) | **`O(n)`** |
| 3. Thấp trước + ô trống (quét) | `O(n^2)` | `O(n)` |
| 4. Ô trống + BIT | `O(n log n)` | `O(n)` |
| 5. Chọn người đầu hàng | `O(n^2)` | `O(n)` |

Với `n <= 2000` mọi cách (trừ vét cạn) đều chạy trong chưa đầy một giây.

## 12. Bộ test

Kiểm tra bằng:

- `is_valid_queue`: kết quả phải thỏa đề (không so cứng).
- Kết quả phải là **hoán vị** của input.
- Oracle vét cạn: với input nhỏ, đếm **mọi** hoán vị hợp lệ; cho thấy luôn có **đúng 1**.
- Đối chiếu năm cách với nhau trên nhiều input.
- Input ngẫu nhiên có seed, sinh bằng cách dựng một hàng thật rồi xáo trộn.

In [21]:
SOLUTIONS = [Solution(), SolutionInsert(), SolutionSlots(), SolutionBIT(), SolutionFront()]


def gen_valid_people(n, hmax, rng):
    """Dựng hàng thật (chiều cao ngẫu nhiên, có thể trùng), tính k, rồi xáo trộn."""
    queue = [rng.randint(0, hmax) for _ in range(n)]
    people = []
    for i, h in enumerate(queue):
        k = sum(1 for j in range(i) if queue[j] >= h)
        people.append([h, k])
    rng.shuffle(people)
    return people


def check(people, got):
    assert is_valid_queue(got), got
    assert sorted(map(tuple, got)) == sorted(map(tuple, people)), "không phải hoán vị của input"


# 12.1 Ví dụ của đề (so cứng + hợp lệ) và edge case
for people, expected in examples:
    for sol in SOLUTIONS:
        got = sol.reconstructQueue([p[:] for p in people])
        assert got == expected, (type(sol).__name__, got)
        check(people, got)

edge = [
    [[0, 0]],                                       # một người, chiều cao 0
    [[5, 0]],
    [[3, 0], [3, 1], [3, 2], [3, 3]],               # cùng chiều cao
    [[1, 0], [2, 0], [3, 0], [4, 0]],               # k toàn 0: xếp tăng dần chiều cao
    [[4, 0], [3, 1], [2, 2], [1, 3]],               # xếp giảm dần chiều cao
    [[10**6, 0], [0, 1], [10**6, 1]],               # chiều cao cực biên
    [[0, 0], [0, 1], [0, 2]],
]
for people in edge:
    results = []
    for sol in SOLUTIONS:
        got = sol.reconstructQueue([p[:] for p in people])
        check(people, got)
        results.append(got)
    assert all(r == results[0] for r in results), (people, results)
    print(people, "->", results[0])
print("OK: ví dụ đề + edge case")

[[0, 0]] -> [[0, 0]]
[[5, 0]] -> [[5, 0]]
[[3, 0], [3, 1], [3, 2], [3, 3]] -> [[3, 0], [3, 1], [3, 2], [3, 3]]
[[1, 0], [2, 0], [3, 0], [4, 0]] -> [[1, 0], [2, 0], [3, 0], [4, 0]]
[[4, 0], [3, 1], [2, 2], [1, 3]] -> [[4, 0], [3, 1], [2, 2], [1, 3]]
[[1000000, 0], [0, 1], [1000000, 1]] -> [[1000000, 0], [0, 1], [1000000, 1]]
[[0, 0], [0, 1], [0, 2]] -> [[0, 0], [0, 1], [0, 2]]
OK: ví dụ đề + edge case


In [22]:
# 12.2 Vét cạn: với input hợp lệ nhỏ, có đúng một hoán vị hợp lệ
rng = random.Random(406)
for _ in range(300):
    n = rng.randint(1, 7)
    people = gen_valid_people(n, rng.choice([2, 4, 10]), rng)
    valid = {tuple(map(tuple, p)) for p in permutations(people) if is_valid_queue(p)}
    assert len(valid) == 1, (people, valid)
    only = [list(x) for x in next(iter(valid))]
    for sol in SOLUTIONS:
        assert sol.reconstructQueue([p[:] for p in people]) == only, (type(sol).__name__, people)
print("OK: 300 input nhỏ — luôn đúng một hoán vị hợp lệ, và cả 5 cách trả về đúng hoán vị đó")

OK: 300 input nhỏ — luôn đúng một hoán vị hợp lệ, và cả 5 cách trả về đúng hoán vị đó


In [23]:
# 12.3 Ngẫu nhiên có seed: n tới 60 — 5 cách thống nhất, hợp lệ
rng = random.Random(4061)
for _ in range(2000):
    n = rng.randint(1, 60)
    people = gen_valid_people(n, rng.choice([1, 3, 10, 1000, 10**6]), rng)
    outs = []
    for sol in SOLUTIONS:
        got = sol.reconstructQueue([p[:] for p in people])
        check(people, got)
        outs.append(got)
    assert all(o == outs[0] for o in outs), people
print("OK: 2000 input ngẫu nhiên n <= 60")

OK: 2000 input ngẫu nhiên n <= 60


In [24]:
# 12.4 Input lớn đúng ràng buộc: n = 2000
rng = random.Random(4062)
for hmax in [3, 100, 10**6]:
    people = gen_valid_people(2000, hmax, rng)
    t = time.perf_counter()
    a = Solution().reconstructQueue([p[:] for p in people])
    dt = time.perf_counter() - t
    b = SolutionBIT().reconstructQueue([p[:] for p in people])
    assert a == b
    check(people, a)
    print(f"n=2000, hmax={hmax:>7}: Cách 2 mất {dt * 1000:.1f} ms; Cách 2 == Cách 4; hợp lệ")

n=2000, hmax=      3: Cách 2 mất 1.5 ms; Cách 2 == Cách 4; hợp lệ


n=2000, hmax=    100: Cách 2 mất 1.0 ms; Cách 2 == Cách 4; hợp lệ


n=2000, hmax=1000000: Cách 2 mất 0.9 ms; Cách 2 == Cách 4; hợp lệ


In [25]:
# 12.5 Phản ví dụ: sắp xếp phụ SAI (k giảm với h giảm) cho kết quả không hợp lệ
def wrong_sort_k_desc(people):
    ans = []
    for h, k in sorted(people, key=lambda p: (-p[0], -p[1])):     # SAI: k giảm dần
        ans.insert(k, [h, k])
    return ans


rng = random.Random(4063)
bad = 0
for _ in range(500):
    people = gen_valid_people(rng.randint(2, 8), 3, rng)
    if not is_valid_queue(wrong_sort_k_desc(people)):
        bad += 1
print("số input mà khóa phụ sai cho kết quả không hợp lệ:", bad, "/ 500")
assert bad > 0
print("ví dụ cụ thể:", wrong_sort_k_desc([[5, 0], [5, 1], [5, 2]]), "(đúng phải là [[5,0],[5,1],[5,2]])")
assert wrong_sort_k_desc([[5, 0], [5, 1], [5, 2]]) != [[5, 0], [5, 1], [5, 2]]

số input mà khóa phụ sai cho kết quả không hợp lệ: 280 / 500
ví dụ cụ thể: [[5, 0], [5, 2], [5, 1]] (đúng phải là [[5,0],[5,1],[5,2]])


## 13. Các lỗi người mới hay gặp

### Lỗi 1: sắp xếp `h` tăng dần rồi chèn

In [26]:
def wrong_ascending(people):
    ans = []
    for h, k in sorted(people, key=lambda p: (p[0], p[1])):       # SAI: thấp trước
        ans.insert(k, [h, k])
    return ans


people = [[7, 0], [4, 4], [7, 1], [5, 0], [6, 1], [5, 2]]
got = wrong_ascending(people)
print(got, "| hợp lệ?", is_valid_queue(got))
assert not is_valid_queue(got)

[[7, 0], [7, 1], [5, 0], [6, 1], [4, 4], [5, 2]] | hợp lệ? False


**Vì sao sai:** khi chèn người thấp trước, những người cao hơn chèn sau sẽ đứng chen vào phía trước người thấp, làm `k` của người thấp bị **tăng lên** (người cao được đếm). Người thấp phải xử lý **sau** cùng. **Cách sửa:** `h` giảm dần.

### Lỗi 2: cùng chiều cao mà `k` giảm dần

Xem `wrong_sort_k_desc` ở mục 12.5. Với 2 người `[[5,0],[5,1]]` lỗi này "tình cờ" vẫn đúng, nhưng với 3 người cùng cao `[[5,0],[5,1],[5,2]]`: chèn `[5,2]` vào hàng rỗng ⇒ `[[5,2]]`; chèn `[5,1]` tại vị trí 1 ⇒ `[[5,2],[5,1]]`; chèn `[5,0]` tại vị trí 0 ⇒ `[[5,0],[5,2],[5,1]]` — người `[5,2]` chỉ có 1 người đứng trước, sai.

In [27]:
people = [[6, 0], [5, 0], [5, 1], [5, 2]]
print(wrong_sort_k_desc(people), "| đúng:", Solution().reconstructQueue([p[:] for p in people]))
bad_found = [p for p in ([[5, 0], [5, 1], [5, 2]], [[6, 0], [5, 0], [5, 1]], [[4, 0], [4, 1], [4, 2], [4, 3]])
             if not is_valid_queue(wrong_sort_k_desc(p))]
print("input làm cách sai hỏng:", bad_found)
assert bad_found

[[5, 0], [6, 0], [5, 1], [5, 2]] | đúng: [[5, 0], [5, 1], [5, 2], [6, 0]]
input làm cách sai hỏng: [[[5, 0], [5, 1], [5, 2]], [[6, 0], [5, 0], [5, 1]], [[4, 0], [4, 1], [4, 2], [4, 3]]]


**Vì sao sai:** nếu chèn `[5,2]` trước khi có `[5,0]`, `[5,1]` thì hàng đang ngắn hơn `2`, `insert(2, ...)` bị "kẹp" về cuối hàng ngắn (vị trí thực tế nhỏ hơn `2`); những người `k` nhỏ chèn sau chỉ đẩy nó thêm một nấc chứ không thể bù đủ. **Cách sửa:** cùng chiều cao thì `k` **tăng dần**.

### Lỗi 3: chèn theo `h` thay vì `k`

In [28]:
def wrong_insert_h(people):
    ans = []
    for h, k in sorted(people, key=lambda p: (-p[0], p[1])):
        ans.insert(h, [h, k])                                    # SAI: phải là k
    return ans


got = wrong_insert_h([[7, 0], [4, 4], [7, 1], [5, 0], [6, 1], [5, 2]])
print(got, "| hợp lệ?", is_valid_queue(got))
assert not is_valid_queue(got)

[[7, 0], [7, 1], [6, 1], [5, 0], [4, 4], [5, 2]] | hợp lệ? False


**Vì sao sai:** `h` là chiều cao, không liên quan đến **vị trí**. Vị trí chèn chính là số người đứng trước, tức `k`. **Cách sửa:** `ans.insert(k, person)`.

### Lỗi 4: đếm cả người thấp hơn, hoặc không đếm người bằng nhau

In [29]:
def wrong_valid_strict(queue):
    for i, (h, k) in enumerate(queue):
        if sum(1 for j in range(i) if queue[j][0] > h) != k:     # SAI: dùng > thay vì >=
            return False
    return True


right = Solution().reconstructQueue([[5, 0], [5, 1], [4, 2]])
print(right, "| hàm hợp lệ đúng:", is_valid_queue(right), "| hàm hợp lệ sai (>):", wrong_valid_strict(right))
assert is_valid_queue(right) and not wrong_valid_strict(right)

[[5, 0], [5, 1], [4, 2]] | hàm hợp lệ đúng: True | hàm hợp lệ sai (>): False


**Vì sao sai:** đề nói "lớn hơn **hoặc bằng**". Nếu dùng `>` thì người cao bằng nhau không được đếm và hàm kiểm tra sẽ bác một đáp án đúng. **Cách sửa:** dùng `>=`.

### Lỗi 5: làm thay đổi input

In [30]:
people = [[7, 0], [4, 4], [7, 1], [5, 0], [6, 1], [5, 2]]
snapshot = [p[:] for p in people]


def inplace_version(people):
    people.sort(key=lambda x: (-x[0], x[1]))                    # sắp xếp tại chỗ: đổi input
    ans = []
    for p in people:
        ans.insert(p[1], p)
    return ans


inplace_version(people)
print("input sau khi gọi:", people, "| còn nguyên?", people == snapshot)
assert people != snapshot

input sau khi gọi: [[7, 0], [7, 1], [6, 1], [5, 0], [5, 2], [4, 4]] | còn nguyên? False


**Vì sao đáng chú ý:** LeetCode chấp nhận, nhưng trong code thật sửa input là tác dụng phụ bất ngờ; thêm nữa các test của ta dùng chung một list sẽ bị nhiễm. **Cách sửa:** `sorted(...)` tạo bản mới (như `Solution` ở mục 9).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu nhận dạng:** mỗi phần tử có hai thuộc tính "giá trị" và "số phần tử lớn hơn/nhỏ hơn đứng trước/sau" ⇒ **sắp xếp theo thuộc tính này, chèn theo thuộc tính kia**. Cùng họ: *Count of Smaller Numbers After Self* (315), *Russian Doll Envelopes* (354), *Number of Visible People in a Queue* (1944).
- **Nguyên tắc "kẻ yếu không làm hại người mạnh":** xử lý từ nhóm mạnh nhất; nhóm yếu hơn không đổi số đếm của nhóm mạnh.
- **Hai thứ tự đối xứng:** cao trước + `insert(k)` hoặc thấp trước + "ô trống thứ `k`". Cùng chiều cao thì thứ tự phụ **ngược nhau** giữa hai cách (`k` tăng ↔ `k` giảm).
- Nếu `n` lớn (`10^5`) và cần `O(n log n)`: BIT hoặc segment tree để tìm ô trống thứ `k`.
- Luôn kiểm bằng hàm hợp lệ trên hàng đợi dựng ngẫu nhiên.

## 15. Biến thể

1. **`k` đếm người cao hơn *chặt chẽ* (`>`):** thứ tự phụ đổi ngược lại và duy nhất không còn được đảm bảo; cần xem lại điều kiện.
2. **`k` đếm người thấp hơn đứng trước:** đối xứng, sắp theo `h` tăng dần rồi chèn ở `k`.
3. **Cho `k` là số người cao hơn đứng *sau*:** đảo hàng rồi dùng cách gốc.
4. **Đếm số cách dựng lại** khi cho phép nhiều đáp án: bài đếm tổ hợp.
5. **Trả về chỉ vị trí của một người** mà không dựng cả hàng.

In [31]:
def reconstruct_shorter_count(people):
    """Biến thể: k = số người THẤP HƠN hoặc bằng đứng trước. Đối xứng: h tăng dần, cùng h thì k tăng, chèn tại k."""
    ans = []
    for h, k in sorted(people, key=lambda p: (p[0], p[1])):
        ans.insert(k, [h, k])
    return ans


def is_valid_shorter(queue):
    return all(sum(1 for j in range(i) if queue[j][0] <= h) == k for i, (h, k) in enumerate(queue))


rng = random.Random(4064)
for _ in range(300):
    n = rng.randint(1, 8)
    heights = [rng.randint(0, 4) for _ in range(n)]
    people = [[h, sum(1 for j in range(i) if heights[j] <= h)] for i, h in enumerate(heights)]
    rng.shuffle(people)
    got = reconstruct_shorter_count(people)
    assert is_valid_shorter(got), (people, got)
print("OK: biến thể 'đếm người thấp hơn hoặc bằng' hợp lệ trên 300 input")
print(reconstruct_shorter_count([[1, 0], [3, 1], [2, 1]]))

OK: biến thể 'đếm người thấp hơn hoặc bằng' hợp lệ trên 300 input
[[1, 0], [3, 1], [2, 1]]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hoán vị | thử mọi thứ tự | `O(n! * n^2)` | `O(n)` | **oracle** cho `n <= 8` |
| 2. Cao trước + `insert(k)` | chèn người cao trước, thấp sau không phá | `O(n^2)` | `O(n)` | **nộp bài**, phỏng vấn |
| 3. Thấp trước + ô trống | đặt vào ô trống thứ `k` | `O(n^2)` | `O(n)` | hiểu đối xứng của ý 2 |
| 4. Ô trống + BIT | tìm ô trống thứ `k` bằng cây Fenwick | `O(n log n)` | `O(n)` | `n` rất lớn |
| 5. Chọn người đầu hàng | người `k = 0` thấp nhất đứng đầu; giảm `k` của người thấp hơn | `O(n^2)` | `O(n)` | góc nhìn "dựng từ trước ra sau" |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `sorted([[5,0],[6,1],[5,2]], key=lambda p: (-p[0], p[1]))`; (b) `a = [1, 2]; a.insert(5, 9); a`; (c) `is_valid_queue([[5,0],[5,1]])`; (d) `is_valid_queue([[4,0],[5,0]])`.

**Bài 2 (tính tay).** Dựng hàng cho `[[6,0],[5,0],[4,0],[3,2],[2,2],[1,4]]` (ví dụ 2) bằng cách chèn từ cao đến thấp, ghi lại hàng sau mỗi lần chèn.

**Bài 3 (điền TODO).** Hoàn thành `reconstruct_todo(people)` theo Cách 2.

In [32]:
def reconstruct_todo(people):
    # TODO 1: sắp xếp people theo (-h, k) -> biến order
    # TODO 2: ans = []
    # TODO 3: với mỗi (h, k) trong order: ans.insert(k, [h, k])
    # TODO 4: return ans
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `reconstruct_slots2(people)` theo Cách 3 (thấp trước, ô trống thứ `k`) **không nhìn lại** mục 7. Chú ý thứ tự phụ khi cùng chiều cao. Kiểm tra bằng `is_valid_queue` trên các ví dụ đề.

**Bài 5 (biến thể / mini challenge).** Viết `count_valid_queues(people)` đếm số hoán vị **hợp lệ** (dùng `permutations` + `is_valid_queue`, set để loại trùng) cho danh sách `people` nhỏ. Với input hợp lệ được dựng từ hàng thật, kết quả luôn `1`. Hãy thử thêm một input **không** hợp lệ như `[[5,0],[5,0]]` và quan sát kết quả `0`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[[6,1],[5,0],[5,2]]`; `[1, 2, 9]`; `True`; `True` (`[4,0]` đứng đầu nên không có ai trước; `[5,0]` chỉ có người thấp hơn `4 < 5` đứng trước nên không được đếm, `k = 0` đúng). Bài 3: xem `reconstruct_insert`.
- **Mức 2 (vừa):** Bài 2: thứ tự xử lý `[6,0],[5,0],[4,0],[3,2],[2,2],[1,4]` (mỗi chiều cao chỉ có một người). `[6,0]` → `[6]`; `[5,0]` chèn `0` → `[5,6]`; `[4,0]` chèn `0` → `[4,5,6]`; `[3,2]` chèn `2` → `[4,5,3,6]`; `[2,2]` chèn `2` → `[4,5,2,3,6]`; `[1,4]` chèn `4` → `[4,5,2,3,1,6]`. Bài 4: sắp theo `(h, -k)`.
- **Mức 3 (sâu):** Bài 5: `len({tuple(map(tuple, p)) for p in permutations(people) if is_valid_queue(p)})`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [33]:
# Bài 1
assert sorted([[5, 0], [6, 1], [5, 2]], key=lambda p: (-p[0], p[1])) == [[6, 1], [5, 0], [5, 2]]
a = [1, 2]
a.insert(5, 9)
assert a == [1, 2, 9]
assert is_valid_queue([[5, 0], [5, 1]]) is True
assert is_valid_queue([[4, 0], [5, 0]]) is True


# Bài 2
people2 = [[6, 0], [5, 0], [4, 0], [3, 2], [2, 2], [1, 4]]
row = []
history = []
for h, k in sorted(people2, key=lambda p: (-p[0], p[1])):
    row.insert(k, [h, k])
    history.append([x[0] for x in row])
assert history[-1] == [4, 5, 2, 3, 1, 6]
assert row == examples[1][1]


# Bài 3
def reconstruct_todo_ans(people):
    order = sorted(people, key=lambda p: (-p[0], p[1]))
    ans = []
    for h, k in order:
        ans.insert(k, [h, k])
    return ans


for people, expected in examples:
    assert reconstruct_todo_ans(people) == expected


# Bài 4
def reconstruct_slots2(people):
    n = len(people)
    ans = [None] * n
    for h, k in sorted(people, key=lambda p: (p[0], -p[1])):
        empties = [i for i in range(n) if ans[i] is None]
        ans[empties[k]] = [h, k]
    return ans


for people, expected in examples:
    got = reconstruct_slots2(people)
    assert got == expected and is_valid_queue(got)


# Bài 5
def count_valid_queues(people):
    return len({tuple(map(tuple, p)) for p in permutations(people) if is_valid_queue(p)})


assert count_valid_queues([[7, 0], [4, 4], [7, 1], [5, 0], [6, 1], [5, 2]]) == 1
assert count_valid_queues([[5, 0], [5, 0]]) == 0
print("OK: toàn bộ đáp án đúng")

OK: toàn bộ đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template** (cao trước, chèn vào vị trí `k`)

```python
class Solution:
    def reconstructQueue(self, people: List[List[int]]) -> List[List[int]]:
        people = sorted(people, key=lambda x: (-x[0], x[1]))
        ans = []
        for p in people:
            ans.insert(p[1], p)
        return ans
```

**Checklist**

- [ ] Sắp xếp `h` **giảm dần**, cùng `h` thì `k` **tăng dần**?
- [ ] Chèn vào vị trí **`k`** (không phải `h`)?
- [ ] Hiểu "`>=`" nghĩa là cả người cao bằng nhau được đếm?
- [ ] Không thay đổi input ngoài ý muốn (dùng `sorted`)?
- [ ] Test bằng hàm kiểm tra hợp lệ, không chỉ so với đáp án ví dụ?
- [ ] Đã thử `n = 1`, mọi người cùng chiều cao, `k` toàn `0`, chiều cao `0` và `10^6`?